# 02 · Train the Conditional Diffusion Model  (Stage 2 — the generative core)

**Crowd-Driven Visual Generation.** Freezes the Stage-1 VAE, encodes each painting to its `[4,16,16]` latent, and trains a **conditional latent DDPM** to denoise those latents given the image's **CLIP embedding** — with classifier-free guidance.

Because CLIP shares one space for images and text, a model trained on *image* embeddings can at inference be steered by a **crowd** embedding aggregated from words/emojis — that's the project's RQ1 novelty. Aggregation happens only at inference (Stage 3), never here.

> Runtime → **GPU (T4)**. Needs the Stage-1 `vae.pt` (pulled from Drive or HF automatically).

## 1. Clone the repo & enter the project

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation

In [ ]:
!pip install -q datasets wandb open-clip-torch   # dataset + monitoring + CLIP encoder
import sys, torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Monitoring & credentials
Runtime prompts — keys stay hidden and are never saved in the notebook.
> **HF token:** a **Write**-scope token is needed for §3 (pulling the VAE from a *private* repo) and §6 (uploading the diffusion checkpoint).

In [ ]:
import getpass

wandb_key = getpass.getpass("W&B API key (https://wandb.ai/authorize — Enter to skip): ").strip()
if wandb_key:
    import wandb; wandb.login(key=wandb_key); print("✓ W&B logged in")
else:
    print("• W&B skipped")

hf_token = getpass.getpass("HF token (Write scope — Enter to skip): ").strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token); print("✓ HuggingFace logged in")
else:
    print("• HF token skipped")

## 3. Drive + locate the Stage-1 VAE
Mounts Drive for checkpoint persistence and finds `vae.pt` — the Drive copy first, else pulled from the HF Hub.

In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/crowdgen/diffusion'   # persisted — survives session end

VAE_CKPT = '/content/drive/MyDrive/crowdgen/vae/vae.pt'
if not os.path.exists(VAE_CKPT):                     # fall back to the HF Hub copy
    from huggingface_hub import hf_hub_download, whoami
    user = whoami()["name"]
    VAE_CKPT = hf_hub_download(f"{user}/crowdgen-vae", "vae.pt")
print("VAE checkpoint:", VAE_CKPT)

## 4. Train the diffusion model
Pre-encodes all latents + CLIP conditions once (VAE & CLIP are frozen), scales latents to ~unit variance, then trains the FiLM-conditioned U-Net (ε-prediction + CFG). W&B logs `loss/eps_mse`, `grad_norm`, `learning_rate`, and a **sample grid** every `--sample-every` epochs (top = real, bottom = generated from that image's CLIP embedding).

Expected on **T4**: ~1–2 min pre-encode + ~8–12 min training (60 epochs).

In [ ]:
!python train_diffusion.py --vae {VAE_CKPT} --dataset huggan/wikiart --limit 5000 \
    --image-size 64 --batch 128 --epochs 60 --lr 2e-4 --out {OUT} \
    --guidance 3.0 --sample-steps 50 \
    --wandb --sample-every 5

## 5. Inspect samples
Top row = real paintings, bottom row = images **generated** from each painting's CLIP embedding. They won't be pixel copies (this is generation, not reconstruction) — look for matching *theme, palette, and composition*. That confirms the conditioning steers the model, which is what makes crowd-conditioning work in Stage 3. *(Also live in W&B.)*

In [ ]:
from IPython.display import Image
Image(f'{OUT}/samples.png')

## 6. Save the checkpoint to the HuggingFace Hub  *(optional)*
Uploads `diffusion.pt` (+ sample grid) to a versioned model repo, alongside the Drive copy. Requires a **Write**-scope token.

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, whoami

REPO_PRIVATE = True
user = whoami()["name"]
repo_id = f"{user}/crowdgen-diffusion"

create_repo(repo_id, repo_type="model", exist_ok=True, private=REPO_PRIVATE)
api = HfApi()
api.upload_file(path_or_fileobj=f"{OUT}/diffusion.pt", path_in_repo="diffusion.pt",
                repo_id=repo_id, repo_type="model")
if os.path.exists(f"{OUT}/samples.png"):
    api.upload_file(path_or_fileobj=f"{OUT}/samples.png", path_in_repo="samples.png",
                    repo_id=repo_id, repo_type="model")
print(f"✓ uploaded to https://huggingface.co/{repo_id}")

## Next — Stage 3: crowd conditioning
Checkpoint at `{OUT}/diffusion.pt` (stores `latent_scale`, `cond_dim`, `unet_ch_mult` for inference). Stage 3 encodes a **crowd** of words/emojis with CLIP, **aggregates** them (mean-pool / cluster-centroid / attention — the RQ1 study) into one condition vector, and runs guided DDIM sampling to turn the crowd into an image.